# Steam Game Recommendation: Playtime-Aware BPR

**CSE 258, Assignment 2**

**Research question:** Does playtime-weighted Bayesian Personalized Ranking (BPR) outperform standard BPR on top-K game recommendation?

The notebook follows the five graded sections:

1. Predictive task and evaluation protocol
2. Dataset, preprocessing and exploratory analysis
3. Modeling
4. Evaluation and results
5. Related work and limitations

<!-- TEAM NOTES (delete before export)
Owners: [S] = Svante, [T] = Tony, [S+T] = both. Only edit cells in your own sections.
Pull before editing, commit small changes often, clear outputs before committing.
See doc.md for the full plan and integration contract. -->

## 0. Setup

In [ ]:
import ast
import gzip
import json
import math
import os
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")
for d in (DATA_DIR, RESULTS_DIR, FIGURES_DIR):
    d.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Predictive task and evaluation protocol  <!-- [S] -->

**Task.** Given a user's observed Steam library and playtime, rank games outside that library. The target is held-out game *ownership*, not future purchases, ratings or hours played.

**Evaluation.** TODO: per-user 80/10/10 split, full-catalog ranking with training (and validation) games masked, macro-averaged Recall@10, Recall@20, NDCG@10 and AUC.

**Baselines.** TODO: popularity, item-based Jaccard, standard BPR.

**Validity.** TODO: validation-only model selection, untouched test set, identical candidate sets across models, fixed seeds.

### 1.1 Evaluation functions  <!-- [S] -->

Defined here so they can be used for BPR early stopping in Section 3. Every model exposes `score_users(user_ids) -> np.ndarray` of shape `(len(user_ids), num_items)`, with scores in shared item-ID order.

For each user, `evaluate` ranks every game they do not own in training (for the test set, validation games are also excluded), then compares the ranking with the user's held-out games:

- **Recall@k:** the fraction of the user's held-out games that appear in the top k.
- **NDCG@10:** like Recall@10, but hits near the top count more (gain $1/\log_2(\text{rank}+1)$), normalized so a perfect ranking scores 1.
- **AUC:** the probability that a random held-out game is scored above a random game the user does not own, over the whole ranking.

Metrics are computed per user and then averaged, so every user counts equally regardless of library size. `sample_eval_users` gives a fixed user subset for fast evaluation during training.

In [ ]:
from scipy.stats import rankdata


def interaction_matrix(dfs, num_users, num_items):
    """Sparse 0/1 user x item matrix from one or more interaction DataFrames."""
    if isinstance(dfs, pd.DataFrame):
        dfs = [dfs]
    users = np.concatenate([df["user_id"].to_numpy() for df in dfs])
    items = np.concatenate([df["item_id"].to_numpy() for df in dfs])
    m = sp.csr_matrix((np.ones(len(users), dtype=np.float32), (users, items)),
                      shape=(num_users, num_items))
    m.data[:] = 1  # duplicates across DataFrames are summed by scipy; reset to 0/1
    return m


def sample_eval_users(eval_df, n_users, seed=SEED):
    """A fixed random subset of users, e.g. for fast per-epoch early stopping."""
    users = np.unique(eval_df["user_id"].to_numpy())
    return np.sort(np.random.default_rng(seed).choice(users, size=min(n_users, len(users)), replace=False))


def evaluate(model, eval_df, mask_dfs, num_users, num_items, ks=(10, 20), ndcg_k=10,
             user_ids=None, batch_size=1024, return_per_user=False):
    """Rank all train-supported items for each eligible user and compute ranking metrics.

    model:     object with score_users(user_ids) -> (len(user_ids), num_items) scores
    eval_df:   held-out interactions (val or test); a user's held-out games are its relevant items
    mask_dfs:  interactions excluded from ranking (train for val; [train, val] for test)
    user_ids:  optional subset of users to evaluate (e.g. from sample_eval_users)

    Eligible users are those with at least one held-out game. For each user:
      Recall@k  = (# held-out games in the top k) / (# held-out games)
      NDCG@10   = DCG@10 / ideal DCG@10, with gain 1 / log2(rank + 1)
      AUC       = probability that a random held-out game is scored above a random
                  candidate the user does not own (ties count 1/2)
    Top-k ties are broken by lower item ID, so results are deterministic.
    Returns macro averages over users, plus per-user metrics if return_per_user=True.
    """
    relevant = interaction_matrix(eval_df, num_users, num_items)
    masked = interaction_matrix(mask_dfs, num_users, num_items)
    assert relevant.multiply(masked).nnz == 0, "held-out items overlap the masked items"

    users = np.flatnonzero(np.diff(relevant.indptr) > 0)  # users with >= 1 held-out game
    if user_ids is not None:
        users = np.intersect1d(users, user_ids)

    max_k = min(max(max(ks), ndcg_k), num_items)
    discounts = 1.0 / np.log2(np.arange(2, max_k + 2))  # gain at rank 1, 2, ...
    ideal_dcg = np.cumsum(discounts[:ndcg_k])

    per_user = []
    for start in range(0, len(users), batch_size):
        batch = users[start:start + batch_size]
        scores = model.score_users(batch)
        if hasattr(scores, "detach"):  # accept torch tensors
            scores = scores.detach().cpu().numpy()
        scores = np.array(scores, dtype=np.float64)
        assert scores.shape == (len(batch), num_items), scores.shape

        rel = relevant[batch].toarray().astype(bool)
        mask = masked[batch].toarray().astype(bool)
        scores[mask] = -np.inf  # owned (training) games can never be recommended
        n_rel = rel.sum(axis=1)
        n_candidates = num_items - mask.sum(axis=1)

        # Top-k: stable sort on -score, so equal scores keep item-ID order.
        top = np.argsort(-scores, axis=1, kind="stable")[:, :max_k]
        hits = np.take_along_axis(rel, top, axis=1)
        row = {"user_id": batch, "n_relevant": n_rel}
        for k in ks:
            row[f"recall@{k}"] = hits[:, :k].sum(axis=1) / n_rel
        dcg = hits[:, :ndcg_k] @ discounts[:hits[:, :ndcg_k].shape[1]]
        row[f"ndcg@{ndcg_k}"] = dcg / ideal_dcg[np.minimum(n_rel, ndcg_k) - 1]

        # AUC via the Mann-Whitney statistic on average ranks (ties count 1/2).
        # Masked items are -inf, so they take the lowest m ranks; subtracting m gives
        # each item's rank among the candidates only.
        ranks = rankdata(scores, axis=1, method="average") - mask.sum(axis=1, keepdims=True)
        n_neg = n_candidates - n_rel
        rank_sum = np.where(rel, ranks, 0).sum(axis=1)
        row["auc"] = (rank_sum - n_rel * (n_rel + 1) / 2) / (n_rel * n_neg)
        per_user.append(pd.DataFrame(row))

    per_user = pd.concat(per_user, ignore_index=True)
    summary = per_user.drop(columns=["user_id", "n_relevant"]).mean().to_dict()
    summary["n_users"] = len(per_user)
    return (summary, per_user) if return_per_user else summary

In [ ]:
# Sanity check on a toy example small enough to verify by hand.
# One user, 6 games. Game 0 is owned in training (masked); games 1 and 4 are held out.
class FixedScores:
    def __init__(self, scores):
        self.scores = np.asarray(scores, dtype=float)

    def score_users(self, user_ids):
        return self.scores[user_ids]


toy_train = pd.DataFrame({"user_id": [0], "item_id": [0]})
toy_heldout = pd.DataFrame({"user_id": [0, 0], "item_id": [1, 4]})
# Candidate ranking by score: game 1 (rank 1), 2, 3, 4 (rank 4), 5. Game 0 is masked.
toy_model = FixedScores([[9.0, 0.9, 0.8, 0.7, 0.6, 0.5]])
toy = evaluate(toy_model, toy_heldout, [toy_train], num_users=1, num_items=6, ks=(1, 3))

# By hand: NDCG@10 = (1/log2(2) + 1/log2(5)) / (1/log2(2) + 1/log2(3)) = 0.877;
# AUC: game 1 beats all 3 negatives, game 4 beats 1 of 3 -> (3 + 1) / 6 = 0.667;
# Recall@1 = 1/2, Recall@3 = 1/2.
expected = {"recall@1": 0.5, "recall@3": 0.5, "ndcg@10": (1 + 1 / np.log2(5)) / (1 + 1 / np.log2(3)),
            "auc": 4 / 6}
for metric, value in expected.items():
    assert np.isclose(toy[metric], value), (metric, toy[metric], value)
print("Toy example matches hand-computed metrics:", {k: round(float(v), 3) for k, v in expected.items()})

## 2. Dataset, preprocessing and exploratory analysis  <!-- [S] -->

**Source.** We use the "User and Item Data" file from Version 1 of the Steam dataset (Pathak, Gupta and McAuley, 2017; Wan and McAuley, 2018; Kang and McAuley, 2018), available from Julian McAuley's dataset page (https://cseweb.ucsd.edu/~jmcauley/datasets.html#steam_data). Steam is Valve's digital storefront and launcher for PC games, where each user has a profile listing the games in their library.

**Contents.** The file has one record per user: their Steam ID, profile URL and the list of games they own. For each game it gives the lifetime playtime (`playtime_forever`, in minutes) and the playtime over the last two weeks. We use only the user ID, game ID and lifetime playtime.

**Collection.** Pathak et al. (2017) built the data by crawling the members of the "GameAus" group, an Australian Steam community, giving 88,310 users and 10,978 games. This matches the raw file exactly. A crawler can only see game libraries that users have made public, and the dataset itself is publicly released.

**Why this matters for the task.**
- *Ownership is imperfect implicit feedback.* Users own games they never play: games received in bundles or as gifts, free-to-play titles and sale purchases. A game missing from a library is not a dislike; the user may never have come across it. That is why the models treat unobserved games as sampled negatives rather than true negatives.
- *Playtime is a noisy preference signal.* Lifetime playtime is heavy-tailed and favours long, replayable games and games owned for longer. This motivates the log transform and per-user normalization tested in Section 3.4.
- *The data is a snapshot.* There are no purchase dates, so we cannot predict future purchases or split the data by time. We hold out a random subset of each user's library and ask whether a model can recover it.
- *Selection bias.* The users are members of a single Australian community group with public profiles, so they may not represent Steam users in general.

### 2.1 Download and parse  <!-- [S] -->

The file has one record per user with a nested list of owned games. We flatten it into one row per (user, game) with the game's lifetime playtime in minutes.

In [ ]:
# The raw file is committed at the repo root; download it only if it is missing.
RAW_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/steam/australian_users_items.json.gz"
RAW_PATH = Path("australian_users_items.json.gz")

if not RAW_PATH.exists():
    print("Downloading (~74 MB)...")
    urllib.request.urlretrieve(RAW_URL, RAW_PATH)

# Parsing takes ~1.5 minutes, so the flattened tables are cached in data/.
RAW_CACHE = DATA_DIR / "raw_interactions.parquet"
USERS_CACHE = DATA_DIR / "raw_users.parquet"
NAMES_CACHE = DATA_DIR / "item_names.parquet"


def parse_user_items(path):
    """Flatten the nested user -> items records into one row per (user, game).

    Each line of the file is a Python dict literal (single quotes), not strict JSON,
    so it is parsed with ast.literal_eval.
    """
    users, items, playtimes = [], [], []
    user_records = []
    item_names = {}
    with gzip.open(path, "rt") as f:
        for line in f:
            record = ast.literal_eval(line)
            user_records.append((record["user_id"], len(record["items"])))
            for it in record["items"]:
                users.append(record["user_id"])
                items.append(it["item_id"])
                playtimes.append(it["playtime_forever"])
                item_names[it["item_id"]] = it["item_name"]
    raw = pd.DataFrame({"user_id": users, "item_id": items, "playtime_forever": playtimes})
    raw_users = pd.DataFrame(user_records, columns=["user_id", "n_items"])
    names = pd.DataFrame(list(item_names.items()), columns=["item_id", "item_name"])
    return raw, raw_users, names


if RAW_CACHE.exists():
    raw = pd.read_parquet(RAW_CACHE)
    raw_users = pd.read_parquet(USERS_CACHE)
    item_names = pd.read_parquet(NAMES_CACHE)
else:
    raw, raw_users, item_names = parse_user_items(RAW_PATH)
    raw.to_parquet(RAW_CACHE, index=False)
    raw_users.to_parquet(USERS_CACHE, index=False)
    item_names.to_parquet(NAMES_CACHE, index=False)

display(pd.Series({
    "user records in file": len(raw_users),
    "unique users": raw_users["user_id"].nunique(),
    "user records with no games": int((raw_users["n_items"] == 0).sum()),
    "interaction rows": len(raw),
    "unique games": raw["item_id"].nunique(),
}, name="raw data").to_frame())
display(raw.head())

### 2.2 Cleaning and filtering  <!-- [S] -->

- **Duplicates.** Some users appear in more than one record, so a (user, game) pair can occur twice. Playtime only grows over time, so we keep the larger value.
- **Zero playtime.** Games that were never played are kept: owning a game is still an implicit signal of interest.
- **Filtering.** We keep users with at least 10 games and games with at least 5 users, repeating until stable, because removing games can push users below the threshold and vice versa. Very sparse users and games give the models too little signal to learn from or be evaluated on.

In [ ]:
MIN_GAMES_PER_USER = 10
MIN_USERS_PER_GAME = 5


def summarize(df):
    """Dataset size statistics for one stage of preprocessing."""
    n_users, n_items = df["user_id"].nunique(), df["item_id"].nunique()
    return {
        "users": n_users,
        "games": n_items,
        "interactions": len(df),
        "density": len(df) / (n_users * n_items),
        "zero-playtime fraction": (df["playtime_forever"] == 0).mean(),
    }


# 1. Duplicates: some users appear in more than one record, so the same (user, game)
#    pair can occur twice. playtime_forever is cumulative, so keep the larger value.
pair_playtimes = raw.groupby(["user_id", "item_id"])["playtime_forever"]
n_conflicting = int((pair_playtimes.nunique() > 1).sum())
clean = pair_playtimes.max().reset_index()
print(f"Removed {len(raw) - len(clean):,} duplicate rows "
      f"({n_conflicting} duplicated pairs had different playtimes; kept the max).")

# 2. Validate playtime and convert minutes to hours. Zero-playtime games are kept:
#    owning a game is still an implicit signal even if it was never played.
assert (clean["playtime_forever"] >= 0).all(), "negative playtime found"
clean["playtime_hours"] = clean["playtime_forever"] / 60


# 3. Iterative k-core filtering: removing games can push users below the threshold
#    and vice versa, so repeat until nothing changes.
def iterative_filter(df, min_games_per_user, min_users_per_game):
    history = [summarize(df)]
    while True:
        n_before = len(df)
        df = df[df.groupby("user_id")["item_id"].transform("size") >= min_games_per_user]
        df = df[df.groupby("item_id")["user_id"].transform("size") >= min_users_per_game]
        history.append(summarize(df))
        if len(df) == n_before:
            break
    return df.reset_index(drop=True), pd.DataFrame(history)


filtered, filter_history = iterative_filter(clean, MIN_GAMES_PER_USER, MIN_USERS_PER_GAME)
filter_history.index = ["after dedup"] + [f"filter pass {i}" for i in range(1, len(filter_history))]
display(filter_history)

### 2.3 Train / validation / test split  <!-- [S] -->

Each user's games are randomly split 80/10/10 (seeded), with at least one validation and one test game per user. The dataset is a snapshot with no purchase dates, so a random split is used rather than a chronological one. Games are indexed from the training split only, since a model cannot score a game it never saw in training.

The split is saved to `data/` so the model sections can reload it without rerunning preprocessing.

In [ ]:
HELDOUT_FRACTION = 0.1  # per user: 10% validation, 10% test, 80% train


def per_user_split(df, seed, heldout_fraction=HELDOUT_FRACTION):
    """Randomly assign each user's games to train/val/test.

    Each user gets n_held = max(1, round(0.1 * n)) validation games and the same number
    of test games; the rest are training games. Users have >= 10 games after filtering,
    so every user has at least 8 training games.
    """
    rng = np.random.default_rng(seed)
    df = df.assign(_r=rng.random(len(df))).sort_values(["user_id", "_r"], kind="stable")
    position = df.groupby("user_id").cumcount().to_numpy()
    n = df.groupby("user_id")["item_id"].transform("size").to_numpy()
    n_held = np.maximum(1, np.floor(heldout_fraction * n + 0.5)).astype(int)
    split = np.where(position < n_held, "test",
                     np.where(position < 2 * n_held, "val", "train"))
    return df.drop(columns="_r").assign(split=split)


split_df = per_user_split(filtered, SEED)

# Contiguous integer IDs. Items are indexed from the training split only: a model cannot
# score a game it never saw in training, so held-out rows for such games are excluded.
user_index = pd.Index(np.sort(split_df["user_id"].unique()))
item_index = pd.Index(np.sort(split_df.loc[split_df["split"] == "train", "item_id"].unique()))

split_df["uid"] = user_index.get_indexer(split_df["user_id"])
split_df["iid"] = item_index.get_indexer(split_df["item_id"])  # -1 = not in training
unsupported = split_df["iid"] < 0
print(f"Excluded {unsupported.sum():,} held-out interactions whose game never appears in training.")
split_df = split_df[~unsupported]


def to_output(part):
    return pd.DataFrame({
        "user_id": part["uid"].astype("int32").to_numpy(),
        "item_id": part["iid"].astype("int32").to_numpy(),
        "playtime_hours": part["playtime_hours"].astype("float32").to_numpy(),
    }).sort_values(["user_id", "item_id"], ignore_index=True)


splits = {name: to_output(split_df[split_df["split"] == name]) for name in ("train", "val", "test")}
num_users, num_items = len(user_index), len(item_index)

# Sanity checks: every user has training data, IDs are contiguous, partitions are disjoint.
assert splits["train"]["user_id"].nunique() == num_users
assert splits["train"]["item_id"].nunique() == num_items
pair_keys = {name: set(zip(s["user_id"], s["item_id"])) for name, s in splits.items()}
assert not (pair_keys["train"] & pair_keys["val"] or pair_keys["train"] & pair_keys["test"]
            or pair_keys["val"] & pair_keys["test"])

total = sum(len(s) for s in splits.values())
display(pd.DataFrame([
    {
        "split": name,
        "interactions": len(s),
        "share": round(len(s) / total, 4),
        "users with >= 1 interaction": s["user_id"].nunique(),
        "users with none (ineligible)": num_users - s["user_id"].nunique(),
    } for name, s in splits.items()
]).set_index("split"))

# Save the shared split (the interface between data/evaluation and model code).
for name, s in splits.items():
    s.to_parquet(DATA_DIR / f"{name}.parquet", index=False)

names_by_id = item_names.set_index("item_id")["item_name"]
meta = {
    "num_users": num_users,
    "num_items": num_items,
    "seed": SEED,
    "heldout_fraction": HELDOUT_FRACTION,
    "min_games_per_user": MIN_GAMES_PER_USER,
    "min_users_per_game": MIN_USERS_PER_GAME,
    "excluded_heldout_interactions": int(unsupported.sum()),
    "user_ids": user_index.tolist(),  # position = integer user_id
    "item_ids": item_index.tolist(),  # position = integer item_id
    "item_names": names_by_id.reindex(item_index).fillna("").tolist(),
}
with open(DATA_DIR / "meta.json", "w") as f:
    json.dump(meta, f)
print(f"Saved train/val/test parquet files and meta.json: {num_users:,} users, {num_items:,} games.")

### 2.4 Exploratory analysis  <!-- [S] -->

Statistics are computed on the filtered dataset (all splits combined). They describe the data only; no model uses them.

In [ ]:
# Descriptive statistics of the filtered dataset (all splits combined; no model uses them).
eda = filtered.merge(item_names, on="item_id", how="left")
games_per_user = eda.groupby("user_id").size()
users_per_game = eda.groupby("item_id").size()
hours = eda["playtime_hours"]

n_users, n_games = len(games_per_user), len(users_per_game)
display(pd.Series({
    "users": f"{n_users:,}",
    "games": f"{n_games:,}",
    "interactions": f"{len(eda):,}",
    "sparsity": f"{1 - len(eda) / (n_users * n_games):.4%}",
    "games per user (median / mean / max)":
        f"{games_per_user.median():.0f} / {games_per_user.mean():.1f} / {games_per_user.max():,}",
    "users per game (median / mean / max)":
        f"{users_per_game.median():.0f} / {users_per_game.mean():.1f} / {users_per_game.max():,}",
    "zero-playtime fraction": f"{(hours == 0).mean():.1%}",
}, name="filtered dataset").to_frame())

quantiles = [0.25, 0.5, 0.75, 0.9, 0.99, 1.0]
display(pd.DataFrame({
    "all interactions": hours.quantile(quantiles),
    "played (> 0 h)": hours[hours > 0].quantile(quantiles),
}).rename(index=lambda q: "max" if q == 1 else f"{q:.0%}").round(1)
  .rename_axis("playtime quantile (hours)"))

# Most-owned games vs games with the most total playtime: popularity and engagement differ.
by_game = eda.groupby(["item_id", "item_name"]).agg(
    owners=("user_id", "size"),
    total_hours=("playtime_hours", "sum"),
    zero_playtime_share=("playtime_hours", lambda h: (h == 0).mean()),
).reset_index(level=0, drop=True)
top_owned = by_game.sort_values("owners", ascending=False).head(10)
top_played = by_game.sort_values("total_hours", ascending=False).head(10)
display(top_owned.style.format({"total_hours": "{:,.0f}", "zero_playtime_share": "{:.0%}"})
        .set_caption("Top 10 games by number of owners"))
display(top_played.style.format({"total_hours": "{:,.0f}", "zero_playtime_share": "{:.0%}"})
        .set_caption("Top 10 games by total playtime"))

# Does a bigger library mean more total playtime? Relevant to the activity-group
# analysis in Section 4.3, which groups users by total playtime.
user_totals = eda.groupby("user_id")["playtime_hours"].sum()
rho = games_per_user.rank().corr(user_totals.rank())
print(f"Spearman correlation between library size and total playtime: {rho:.2f}")

In [ ]:
import matplotlib.ticker  # noqa: E402  (tick formatting)

# Plot style: one series per panel, recessive axes and grid.
BLUE, INK, MUTED, GRID = "#2a78d6", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
})


def log_hist(ax, values, xlabel, title):
    """Histogram with log-spaced bins on log-log axes, for long-tailed counts."""
    bins = np.logspace(np.log10(values.min()), np.log10(values.max()), 40)
    ax.hist(values, bins=bins, color=BLUE, edgecolor="white", linewidth=0.6)
    ax.set(xscale="log", yscale="log", xlabel=xlabel, ylabel="count", title=title)
    ax.axvline(values.median(), color=INK, linestyle="--", linewidth=1)
    ax.annotate(f"median {values.median():.0f}", xy=(values.median(), 1),
                xycoords=("data", "axes fraction"), xytext=(4, -14),
                textcoords="offset points", color=INK, fontsize=9)


fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
log_hist(axes[0], games_per_user, "games in library", "Games per user")
log_hist(axes[1], users_per_game, "number of owners", "Users per game")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_degree_distributions.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(hours, bins=60, color=BLUE, edgecolor="white", linewidth=0.6)
axes[0].set(yscale="log", xlabel="playtime (hours)", ylabel="interactions (log scale)",
            title="Raw playtime")
played = hours[hours > 0]
axes[1].hist(np.log1p(played), bins=60, color=BLUE, edgecolor="white", linewidth=0.6)
axes[1].set(xlabel="log(1 + playtime in hours)", ylabel="interactions",
            title="Log-transformed playtime (played games only)")
axes[1].yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
axes[1].text(0.98, 0.95, f"not shown: {(hours == 0).sum():,} zero-playtime\ninteractions ({(hours == 0).mean():.0%})",
             transform=axes[1].transAxes, ha="right", va="top", color=MUTED, fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_playtime.png", bbox_inches="tight")
plt.show()

**Observations.**

- *Long tails on both sides.* The median user owns 54 games, but some own thousands; the median game has 70 owners, while the most popular have over 40,000. Popularity alone will be a strong baseline, and models must do better than recommending the same few games to everyone.
- *Playtime is extremely skewed.* 36% of owned games were never played, the median is under an hour, and the top 1% exceed ~300 hours. Raw hours would let a handful of interactions dominate a weighted loss; after `log(1 + t)` the played interactions spread out smoothly, which motivates the log weighting in Section 3.4.
- *Ownership is not the same as engagement.* The most-owned and most-played games overlap only partly. Some heavily owned entries, such as *Dota 2 Test* and *Left 4 Dead 2 Beta*, are test or beta builds granted automatically to owners of the main game, and over 90% of their owners never played them.
- *Activity is confounded with library size.* Users with larger libraries also play more in total (Spearman correlation 0.61), so the activity groups in Section 4.3 partly reflect library size.

## 3. Modeling  <!-- [S+T] -->

TODO: formulation (inputs, outputs, objective) and the trade-offs between the models (complexity, efficiency, implementation challenges).

### 3.0 Load the shared split  <!-- [S] -->

In [ ]:
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_DIR / "val.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")
with open(DATA_DIR / "meta.json") as f:
    meta = json.load(f)
num_users, num_items = meta["num_users"], meta["num_items"]

### 3.1 Popularity baseline  <!-- [S] -->

Score each game by its number of distinct training users.

In [ ]:
class Popularity:
    def __init__(self, train_df, num_items):
        raise NotImplementedError  # TODO [S]

    def score_users(self, user_ids):
        raise NotImplementedError  # TODO [S]

### 3.2 Item-based Jaccard similarity  <!-- [S] -->

$J(i,j)=\frac{|U_i\cap U_j|}{|U_i\cup U_j|}$. A candidate's score is the sum of its similarities to the games in the user's training history.

In [ ]:
class ItemJaccard:
    def __init__(self, train_df, num_users, num_items, top_k_neighbors=100):
        raise NotImplementedError  # TODO [S]

    def score_users(self, user_ids):
        raise NotImplementedError  # TODO [S]

### 3.3 Standard BPR  <!-- [T] -->

$s(u,i)=p_u^\top q_i$, trained on triplets $(u,i,j)$ with $i$ a training-owned game and $j$ a sampled game outside the user's training history.

$\mathcal L=\mathbb E[-\log\sigma(s(u,i)-s(u,j))]+\lambda\|\Theta\|_2^2$

In [ ]:
class BPR(torch.nn.Module):
    def __init__(self, num_users, num_items, dim=64):
        super().__init__()
        raise NotImplementedError  # TODO [T]

    def score_users(self, user_ids):
        raise NotImplementedError  # TODO [T]: return a numpy array (n_users, num_items)


def train_bpr(train_df, val_df, num_users, num_items, weights=None, dim=64, lr=1e-3,
              reg=1e-4, batch_size=4096, epochs=30, seed=SEED):
    """Train BPR with optional per-interaction weights (None = standard BPR).
    Uses softplus(s_neg - s_pos) and early stopping on validation NDCG@10."""
    raise NotImplementedError  # TODO [T]

### 3.4 Playtime-weighted BPR  <!-- [T] -->

$\mathcal L=\mathbb E[-w_{ui}\log\sigma(s(u,i)-s(u,j))]+\lambda\|\Theta\|_2^2$, with $a_{ui}=(1+\log(1+t_{ui}))^\alpha$.

| Variant | Raw weight |
|---|---|
| Standard BPR | 1 |
| Log-weighted BPR | $a_{ui}$ |
| User-normalized BPR | $a_{ui}/\operatorname{mean}_{k\in I_u^{train}}a_{uk}$ |

Weights are normalized to mean 1 over training interactions.

In [ ]:
def playtime_weights(train_df, alpha=1.0, user_normalize=False):
    """Return one weight per training interaction, normalized to mean 1."""
    raise NotImplementedError  # TODO [T]

## 4. Evaluation and results  <!-- [S+T] -->

TODO: why these metrics suit the task, and how the results show whether playtime weighting helps.

### 4.1 Experiment A: main comparison (test set)  <!-- [S+T] -->

In [ ]:
# TODO: train/fit all five models, evaluate on test, show the results table,
# and save results/main_comparison.csv (model, seed, split, metrics, config).

### 4.2 Experiment B: playtime sensitivity (alpha sweep, validation)  <!-- [T] -->

In [ ]:
# TODO [T]: alpha in {0, 0.25, 0.5, 1, 2}; plot validation NDCG@10 vs alpha.

### 4.3 Experiment C: user activity groups  <!-- [S+T] -->

In [ ]:
# TODO: split users by total training playtime (bottom 25%, middle 50%, top 25%),
# report group sizes and per-group metrics for standard vs weighted BPR.

## 5. Related work and limitations  <!-- [S+T] -->

- Rendle et al. (2009), *BPR: Bayesian Personalized Ranking from Implicit Feedback*. TODO [T]
- Hu, Koren and Volinsky (2008), *Collaborative Filtering for Implicit Feedback Datasets*. TODO [T]
- Pathak, Gupta and McAuley (2017), *Generating and Personalizing Bundle Recommendations on Steam*. TODO [S]

TODO: how our results compare with these works, and the limitations of the study.